<a href="https://colab.research.google.com/github/GuiMcs00/asian-nlp-lab/blob/main/notebooks/para-quebrar-o-gelo/ambiente_para_contato_inicial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oi Iasmin!
Estou fazendo esse notebook pra gente quebrar o gelo com o modelo que vamos treinar depois. Vou aproveitar e (tentar kkk) explicar o suficiente pra fazer sentido o que está acontecendo aqui. qlqr coisa manda mensagem se precisar.
Aproveitando pra dizer: esse é um notebook onde podemos criar células de código python e código texto markdown (como esse). Você pode executar tudo no botão em cima dessa célula para executar todas as células executáveis de cima para baixo, ou pode executar uma por vez clicando no 'play' que aparece no canto quando passa o mouse em cima da célula executável. (Adianto que é importante executar a célula abaixo primeiro, depois pode executar qualquer uma).

> OBS:
> Eu estou fazendo um repositório com esse mesmo código basicamente. No futuro pode ser que usemos. [Clica aqui que te levo no meu repo](https://github.com/GuiMcs00/asian-nlp-lab)

In [ ]:
# Instalando as ferramentas necessárias pro nosso laboratório funfar
!pip install transformers datasets evaluate accelerate scikit-learn pandas

## Tokenização multilíngue com XLM-RoBERTa

Na próxima célula, estamos carregando o `AutoTokenizer` do modelo `FacebookAI/xlm-roberta-base` e aplicá-lo a frases em diferentes idiomas. O objetivo é observar como um tokenizer multilíngue representa entradas em coreano, português, japonês e chinês antes de qualquer etapa de treinamento ou classificação.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

texts = [
    "안녕하세요",
    "Eu estou estudando coreano",
    "今日は日本語を勉強します",
    "我正在学习中文"
]

tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

tokens

{'input_ids': tensor([[     0, 107687,      2,      1,      1,      1,      1,      1],
        [     0,   5177,  30472, 120175,    557,  56458,   3922,      2],
        [     0, 100752,  98449,    251,  40554,   5182,      2,      1],
        [     0,  13129,  11560,   7272,  32095,      2,      1,      1]]), 'attention_mask': tensor([[1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 0, 0]])}

## Inspeção de tokenização

A próxima célula define uma lista de exemplos em coreano e, para cada frase, chama `tokenizer.tokenize()` para mostrar a segmentação em subwords feita pelo XLM-RoBERTa. Ela imprime a frase original seguida da lista de tokens — útil para entender como o tokenizer separa palavras, pontuação e morfemas antes de seguir para vetorização ou modelos de classificação.

In [ ]:
examples = [
    "눈",
    "눈이 와요.",
    "눈이 아파요.",
    "저는 한국어를 공부해요."
]

for text in examples:
    encoded = tokenizer.tokenize(text)
    print(text)
    print(encoded)
    print()

눈
['▁눈']

눈이 와요.
['▁눈', '이', '▁와', '요', '.']

눈이 아파요.
['▁눈', '이', '▁아', '파', '요', '.']

저는 한국어를 공부해요.
['▁저는', '▁한국어', '를', '▁공부', '해요', '.']



## Exemplo simples de dataset de classificação

Este é um exemplo minimalista de dataset para tarefas de classificação: duas chaves (`ko` e `not_ko`) contendo listas de frases.
Serve apenas para demonstrar como buscar e identificar a classe (chave) associada a um texto de exemplo.

In [2]:
initial_dataset = {"ko": ["안녕하세요", "저는 학생입니다", "눈이 와요"],
                    "not_ko": ["Eu gosto de estudar idiomas", "こんにちは", "你好"]}

input_text = input("digita ai pa nois: ")

for key, values in initial_dataset.items():
    if input_text in values:
        print(input_text, "=>", key)
        break
else:
    print("O texto não está presente no dataset.")

안녕하세요 => ko


## Classificação por regra (detecção de Hangul)

A próxima célula implementa um classificador muito simples baseado em regras: a função `contains_hangul` verifica se um texto contém caracteres Hangul (intervalo Unicode `U+AC00`–`U+D7A3`) e `predict_language_rule_based` retorna `"ko"` se houver Hangul ou `"not_ko"` caso contrário. Em seguida, o código solicita uma entrada do usuário e imprime a predição no formato `entrada => predição`. Este método é apenas ilustrativo e serve para mostrar uma heurística rápida antes de usar modelos estatísticos ou de ML.

In [ ]:
def contains_hangul(text):
    return any('\uac00' <= char <= '\ud7a3' for char in text)

def predict_language_rule_based(text):
    if contains_hangul(text):
        return "ko"
    return "not_ko"


input_text = input("digita ai pa nois dnv: ")
print(input_text, "=>", predict_language_rule_based(input_text))


digita ai pa nois dnv: は
は => not_ko
